# Week 3 — Model Training: NLLB-200 for Dholuo (Steve)

**Assignment:** mT5 / NLLB — Dholuo, no new-language workaround needed (NLLB natively supports Dholuo as `luo_Latn`).

**Week 3 requirements this notebook satisfies:**
1. Train for **10 epochs** using **batch training** (`Seq2SeqTrainer` + `per_device_train_batch_size`, dynamic batch padding via `DataCollatorForSeq2Seq`).
2. `save_strategy="epoch"` — checkpoints saved automatically after every epoch, no manual saving.
3. **Google Drive mounted first** — checkpoints and final model are written straight to Drive so a Colab disconnect (as happened in Week 2) doesn't cost the run.

Base model: `facebook/nllb-200-distilled-600M`. Since Dholuo (`luo_Latn`) is already in NLLB's pretrained vocabulary, this is a **straightforward fine-tune** — no new token, no embedding resize, no LoRA workaround required (unlike the Ekegusii track).

## Step 1: Mount Google Drive

Do this **before anything else** so every checkpoint this notebook produces is written directly to Drive, not to the ephemeral Colab disk.

In [19]:
from google.colab import drive
drive.mount('/content/drive')

import os

PROJECT_DIR = "/content/drive/MyDrive/psa-dholuo-mt"
DATA_DIR = os.path.join(PROJECT_DIR, "data")
CHECKPOINT_DIR = os.path.join(PROJECT_DIR, "nllb-dholuo-checkpoints")
FINAL_MODEL_DIR = os.path.join(PROJECT_DIR, "nllb-dholuo-final")

for d in [PROJECT_DIR, DATA_DIR, CHECKPOINT_DIR, FINAL_MODEL_DIR]:
    os.makedirs(d, exist_ok=True)

print("✅ Drive mounted. Project folder ready at:", PROJECT_DIR)
print("   Checkpoints will be saved to:", CHECKPOINT_DIR)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✅ Drive mounted. Project folder ready at: /content/drive/MyDrive/psa-dholuo-mt
   Checkpoints will be saved to: /content/drive/MyDrive/psa-dholuo-mt/nllb-dholuo-checkpoints


## Step 2: Install Required Libraries

**Note:** earlier drafts of this notebook (and some teammates' Ekegusii notebooks) pinned an old `transformers==4.38.2`. On the current Colab image that old pin now conflicts with newer transitive dependencies (`tokenizers`, `huggingface_hub`, `safetensors`) and throws `ImportError: cannot import name 'EncoderDecoderCache' from 'transformers'` when `Seq2SeqTrainer` is imported. The fix is to **not** force an old version — install the current compatible set instead and let pip resolve matching versions together.

In [20]:
!pip install -q -U transformers datasets accelerate evaluate sacrebleu sentencepiece

import transformers
print("transformers version:", transformers.__version__)
print("If Step 3's import still raises an ImportError, go to Runtime > Restart session,")
print("then re-run Step 1 (mount Drive) and continue from Step 3 — no need to re-run this cell.")

transformers version: 5.14.1
If Step 3's import still raises an ImportError, go to Runtime > Restart session,
then re-run Step 1 (mount Drive) and continue from Step 3 — no need to re-run this cell.


## Step 3: Import Libraries

In [21]:
import gc
import glob

import numpy as np
import pandas as pd
import torch
from datasets import Dataset
from sklearn.model_selection import train_test_split
from transformers import (
    AutoTokenizer,
    AutoModelForSeq2SeqLM,
    DataCollatorForSeq2Seq,
    Seq2SeqTrainingArguments,
    Seq2SeqTrainer,
)
import evaluate

torch.cuda.empty_cache()
gc.collect()

print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("⚠️ No GPU detected — go to Runtime > Change runtime type > T4 GPU before training.")

PyTorch version: 2.11.0+cu128
CUDA available: True
GPU: Tesla T4


## Step 4: Load the Dholuo Dataset

The dataset used here is `data/processed/psa_dataset_dholuo_somali.csv` from the shared project repo (English / Kiswahili / Dholuo / Somali PSA pairs).

**Before running the next cell**, upload that CSV to your Drive at:
`MyDrive/psa-dholuo-mt/data/` — then set `DHOLUO_CSV_FILENAME` below to match its exact name (e.g. if Drive renamed it on upload).

The cell **copies the file to local Colab disk before reading it**, rather than reading it directly off the Drive mount. Reading repeatedly through the Drive FUSE mount — especially right after a file was just uploaded via Drive's web UI and hasn't fully synced — can hang for a long time with no error and no progress. A one-time local copy avoids that entirely. If the file isn't found at all, it falls back to a manual upload prompt.

In [22]:
import shutil

DHOLUO_CSV_FILENAME = "psa_dataset_dholuo_somali_copy.csv"  # <- change this if your file on Drive has a different name

drive_csv_path = os.path.join(DATA_DIR, DHOLUO_CSV_FILENAME)
local_csv_path = "/content/" + DHOLUO_CSV_FILENAME  # local disk copy -- much faster/more reliable to read than the Drive mount directly

if os.path.exists(drive_csv_path):
    print(f"📥 Found on Drive: {drive_csv_path}")
    print("   Copying to local Colab disk first, then reading from there...")
    shutil.copy(drive_csv_path, local_csv_path)
    df = pd.read_csv(local_csv_path)
    print(f"✅ Loaded {df.shape[0]:,} rows")
else:
    print(f"⚠️ Not found at {drive_csv_path} — check DHOLUO_CSV_FILENAME above, or falling back to manual upload.")
    from google.colab import files
    uploaded = files.upload()
    filename = list(uploaded.keys())[0]
    df = pd.read_csv(filename)
    shutil.copy(filename, drive_csv_path)  # keep a copy on Drive for next time
    print(f"✅ Loaded {filename} and saved a copy to {drive_csv_path}")

print("\nDataset shape:", df.shape)
df.head(3)

📥 Found on Drive: /content/drive/MyDrive/psa-dholuo-mt/data/psa_dataset_dholuo_somali_copy.csv
   Copying to local Colab disk first, then reading from there...
✅ Loaded 16,029 rows

Dataset shape: (16029, 8)


,PSA_Id,Domain,English,Kiswahili,Dholuo,Somali,Class,Source
0,1,Education,Comprehensive COVID-19 health and safety proto...,Itifaki kamili za afya na usalama za COVID-19 ...,Chenro mag thieth: Chenro mag thieth kod ritru...,Barnaamijyada caafimaadka iyo amniga ee COVID-...,PSA,original_baseline_dataset
1,2,Education,Digital learning platform providing free educa...,Jukwaa la kujifunza kidijitali linalotoa maudh...,Kenya Education Cloud: Ohinga mar somo mar dij...,Barashada dhijitaalka ah ee bixisa waxyaabaha ...,PSA,original_baseline_dataset
2,3,Education,KUCCPS portal will open in March 2025 for univ...,Lango la KUCCPS litafunguliwa Machi 2025 kwa n...,KUCCPS Portal: KUCCPS portal biro yawore e dwe...,Gudaha KUCCPS waxaa la furi doonaa bishii Maar...,PSA,original_baseline_dataset


## Step 5: Inspect the Dataset

In [23]:
print(df.info())
print("\nMissing values:\n", df.isnull().sum())
print("\nColumns:", df.columns.tolist())
print("\nDomain value counts:\n", df["Domain"].value_counts())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 16029 entries, 0 to 16028
Data columns (total 8 columns):
 #   Column     Non-Null Count  Dtype 
---  ------     --------------  ----- 
 0   PSA_Id     16029 non-null  int64 
 1   Domain     16029 non-null  object
 2   English    16029 non-null  object
 3   Kiswahili  16029 non-null  object
 4   Dholuo     16029 non-null  object
 5   Somali     5111 non-null   object
 6   Class      16029 non-null  object
 7   Source     16029 non-null  object
dtypes: int64(1), object(7)
memory usage: 1001.9+ KB
None

Missing values:
 PSA_Id           0
Domain           0
English          0
Kiswahili        0
Dholuo           0
Somali       10918
Class            0
Source           0
dtype: int64

Columns: ['PSA_Id', 'Domain', 'English', 'Kiswahili', 'Dholuo', 'Somali', 'Class', 'Source']

Domain value counts:
 Domain
Health               3764
Agriculture          3645
Education            2844
Governance           2748
Security             1795
Securit

## Step 6: Clean the Dataset

Light cleaning consistent with the Week 2 report (this raw file predates the fully-validated `psa_train/dev/test.csv` splits Selmah's pipeline produces, so a few of those same fixes are re-applied here):

- Keep only the columns this task needs (English, Kiswahili, Dholuo, plus `PSA_Id`/`Domain` for traceability).
- Merge the `"Security"` / `"Security & Safety"` domain-label inconsistency noted in the Week 1 report.
- Strip whitespace, drop exact duplicate rows and any row with an empty English/Kiswahili/Dholuo field.

In [24]:
clean_df = df[["PSA_Id", "Domain", "English", "Kiswahili", "Dholuo"]].copy()

# Merge the known domain-label inconsistency
clean_df["Domain"] = clean_df["Domain"].replace({"Security": "Security & Safety"})

# Strip whitespace on the text columns
for col in ["English", "Kiswahili", "Dholuo"]:
    clean_df[col] = clean_df[col].astype(str).str.strip()

before = len(clean_df)

# Drop empties
clean_df = clean_df[
    (clean_df["English"] != "") & (clean_df["Kiswahili"] != "") & (clean_df["Dholuo"] != "")
]

# Drop exact duplicate rows
clean_df = clean_df.drop_duplicates(subset=["English", "Kiswahili", "Dholuo"]).reset_index(drop=True)

after = len(clean_df)
print(f"Rows before cleaning: {before:,}")
print(f"Rows after cleaning:  {after:,}  (removed {before - after})")
clean_df.head(3)

Rows before cleaning: 16,029
Rows after cleaning:  16,029  (removed 0)


,PSA_Id,Domain,English,Kiswahili,Dholuo
0,1,Education,Comprehensive COVID-19 health and safety proto...,Itifaki kamili za afya na usalama za COVID-19 ...,Chenro mag thieth: Chenro mag thieth kod ritru...
1,2,Education,Digital learning platform providing free educa...,Jukwaa la kujifunza kidijitali linalotoa maudh...,Kenya Education Cloud: Ohinga mar somo mar dij...
2,3,Education,KUCCPS portal will open in March 2025 for univ...,Lango la KUCCPS litafunguliwa Machi 2025 kwa n...,KUCCPS Portal: KUCCPS portal biro yawore e dwe...


## Step 7: Build English→Dholuo and Kiswahili→Dholuo Pairs (Leakage-Safe Split)

Following the same multilingual pattern the rest of the group uses for Ekegusii: every PSA row becomes **two** training examples (`English → Dholuo` and `Kiswahili → Dholuo`).

The split is done **before** exploding into pairs, and it's grouped by the Dholuo target text first (not just `PSA_Id`) — this guarantees that if two different PSAs happen to share an identical Dholuo translation, both copies land in the same split, so no target sentence leaks between train and test.

In [25]:
SRC_LANG_CODES = {"English": "eng_Latn", "Kiswahili": "swh_Latn"}
TARGET_LANG = "luo_Latn"  # Dholuo — already native to NLLB-200, no new token needed

# Group PSA_Ids by identical Dholuo target text so duplicated targets can't
# end up split across train/val/test
groups = clean_df.groupby("Dholuo")["PSA_Id"].apply(list)
unique_targets = groups.index.tolist()

train_targets, temp_targets = train_test_split(unique_targets, test_size=0.20, random_state=42)
val_targets, test_targets = train_test_split(temp_targets, test_size=0.50, random_state=42)

def targets_to_ids(target_list):
    ids = []
    for t in target_list:
        ids.extend(groups[t])
    return set(ids)

train_ids = targets_to_ids(train_targets)
val_ids = targets_to_ids(val_targets)
test_ids = targets_to_ids(test_targets)

assert train_ids.isdisjoint(val_ids) and train_ids.isdisjoint(test_ids) and val_ids.isdisjoint(test_ids)
print(f"PSA_Ids -> train: {len(train_ids):,}, val: {len(val_ids):,}, test: {len(test_ids):,}")

def build_pairs(id_set):
    subset = clean_df[clean_df["PSA_Id"].isin(id_set)]
    pairs = []
    for _, row in subset.iterrows():
        for src_col, src_lang in SRC_LANG_CODES.items():
            pairs.append({
                "source_text": row[src_col],
                "target_text": row["Dholuo"],
                "source_lang": src_lang,
                "domain": row["Domain"],
            })
    return pd.DataFrame(pairs)

train_df = build_pairs(train_ids)
val_df = build_pairs(val_ids)
test_df = build_pairs(test_ids)

train_dataset = Dataset.from_pandas(train_df.reset_index(drop=True))
val_dataset = Dataset.from_pandas(val_df.reset_index(drop=True))
test_dataset = Dataset.from_pandas(test_df.reset_index(drop=True))

print(f"✅ Train pairs: {len(train_dataset):,}, Val pairs: {len(val_dataset):,}, Test pairs: {len(test_dataset):,}")

train_t, val_t, test_t = set(train_df["target_text"]), set(val_df["target_text"]), set(test_df["target_text"])
print("Overlapping Dholuo targets train/val:", len(train_t & val_t))
print("Overlapping Dholuo targets train/test:", len(train_t & test_t))

PSA_Ids -> train: 12,827, val: 1,600, test: 1,602
✅ Train pairs: 25,654, Val pairs: 3,200, Test pairs: 3,204
Overlapping Dholuo targets train/val: 0
Overlapping Dholuo targets train/test: 0


## Step 8: Save the Splits to Drive

Saving these now means the exact train/val/test split used for training is reproducible and survives a runtime disconnect, and your teammates or the instructor can inspect it later.

In [26]:
train_df.to_csv(os.path.join(DATA_DIR, "dholuo_train.csv"), index=False)
val_df.to_csv(os.path.join(DATA_DIR, "dholuo_val.csv"), index=False)
test_df.to_csv(os.path.join(DATA_DIR, "dholuo_test.csv"), index=False)
print("✅ Splits saved to", DATA_DIR)

✅ Splits saved to /content/drive/MyDrive/psa-dholuo-mt/data


## Step 9: Verify the Splits

In [27]:
from collections import Counter

print("Train source_lang distribution:", Counter(train_dataset["source_lang"]))
print("Val source_lang distribution:  ", Counter(val_dataset["source_lang"]))
print("Test source_lang distribution: ", Counter(test_dataset["source_lang"]))

print("\nExample training pair:")
print(train_dataset[0])

Train source_lang distribution: Counter({'eng_Latn': 12827, 'swh_Latn': 12827})
Val source_lang distribution:   Counter({'eng_Latn': 1600, 'swh_Latn': 1600})
Test source_lang distribution:  Counter({'eng_Latn': 1602, 'swh_Latn': 1602})

Example training pair:
{'source_text': 'Comprehensive COVID-19 health and safety protocols for school reopening including mandatory masks and temperature monitoring', 'target_text': "Chenro mag thieth: Chenro mag thieth kod ritruok kuom COVID-19 ma oriwo tiyo gi maske kod ng'iyo chal mar liet e kinde ma skul chako tiyo kendo", 'source_lang': 'eng_Latn', 'domain': 'Education'}


## Step 10: Load the NLLB-200 Tokenizer and Model

Dholuo (`luo_Latn`) is already part of NLLB-200's 200+ language vocabulary, so we load the tokenizer and model as-is — no `add_tokens`, no embedding resize, no warm-starting. The assertion below confirms `luo_Latn` resolves to a real token rather than `<unk>`.

In [28]:
MODEL_NAME = "facebook/nllb-200-distilled-600M"

print(f"📦 Loading tokenizer and model: {MODEL_NAME} ...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME)

luo_id = tokenizer.convert_tokens_to_ids(TARGET_LANG)
assert luo_id != tokenizer.unk_token_id, f"{TARGET_LANG} was not found in the tokenizer vocabulary!"
print(f"✅ '{TARGET_LANG}' resolves to token id {luo_id} (native NLLB language, not an added token)")

# Tell the model's generation config which language to force as the first
# generated token — needed for predict_with_generate during eval/training
model.generation_config.forced_bos_token_id = luo_id

if torch.cuda.is_available():
    model = model.to("cuda")

model.gradient_checkpointing_enable()  # trades a little speed for a lot of memory headroom
print("✅ Model loaded and moved to GPU (if available), gradient checkpointing enabled")

📦 Loading tokenizer and model: facebook/nllb-200-distilled-600M ...


Loading weights:   0%|          | 0/512 [00:00<?, ?it/s]

✅ 'luo_Latn' resolves to token id 256111 (native NLLB language, not an added token)
✅ Model loaded and moved to GPU (if available), gradient checkpointing enabled


## Step 11: Tokenize the Datasets

Each example's `source_lang` is set per-row (`eng_Latn` or `swh_Latn`) before encoding the source text, and `tokenizer.tgt_lang = "luo_Latn"` is set once so NLLB's own tokenizer handles the target-language tagging natively. Source and target are tokenized together in one call using the `text_target=` argument — the older `with tokenizer.as_target_tokenizer():` pattern is deprecated in current `transformers` versions, so `text_target=` is used instead.

In [29]:
MAX_LEN = 128

def tokenize_batch(examples):
    tokenizer.tgt_lang = TARGET_LANG
    input_ids_list, attention_mask_list, labels_list = [], [], []

    for src_text, tgt_text, src_lang in zip(
        examples["source_text"], examples["target_text"], examples["source_lang"]
    ):
        tokenizer.src_lang = src_lang
        enc = tokenizer(
            src_text,
            text_target=tgt_text,
            max_length=MAX_LEN,
            truncation=True,
        )
        input_ids_list.append(enc["input_ids"])
        attention_mask_list.append(enc["attention_mask"])
        labels_list.append(enc["labels"])

    return {
        "input_ids": input_ids_list,
        "attention_mask": attention_mask_list,
        "labels": labels_list,
    }

print("📝 Tokenizing train/val/test splits...")
train_tokenized = train_dataset.map(tokenize_batch, batched=True, batch_size=32, remove_columns=train_dataset.column_names)
val_tokenized = val_dataset.map(tokenize_batch, batched=True, batch_size=32, remove_columns=val_dataset.column_names)
test_tokenized = test_dataset.map(tokenize_batch, batched=True, batch_size=32, remove_columns=test_dataset.column_names)
print("✅ Tokenization complete")

# Sanity check: how many sequences hit the max-length cap and got truncated?
def count_truncated(raw_dataset, col):
    return sum(1 for t in raw_dataset[col] if len(tokenizer(t)["input_ids"]) >= MAX_LEN)

print(f"Train source sequences hitting the {MAX_LEN}-token cap: {count_truncated(train_dataset, 'source_text')} / {len(train_dataset)}")
print(f"Train target sequences hitting the {MAX_LEN}-token cap: {count_truncated(train_dataset, 'target_text')} / {len(train_dataset)}")

📝 Tokenizing train/val/test splits...


Map:   0%|          | 0/25654 [00:00<?, ? examples/s]

Map:   0%|          | 0/3200 [00:00<?, ? examples/s]

Map:   0%|          | 0/3204 [00:00<?, ? examples/s]

✅ Tokenization complete
Train source sequences hitting the 128-token cap: 4 / 25654
Train target sequences hitting the 128-token cap: 16 / 25654


## Step 12: Data Collator (Batch Training)

`DataCollatorForSeq2Seq` dynamically pads each **batch** to the longest sequence in that batch (rather than padding the whole dataset to one fixed length), and pads labels with `-100` so padding tokens are ignored in the loss. This — combined with `per_device_train_batch_size` on `Seq2SeqTrainingArguments` below — is what satisfies the batch-training requirement.

In [30]:
data_collator = DataCollatorForSeq2Seq(tokenizer=tokenizer, model=model, padding=True)
print("✅ Data collator ready — batches will be dynamically padded per-batch, not per-dataset")

✅ Data collator ready — batches will be dynamically padded per-batch, not per-dataset


## Step 13: Evaluation Metrics

Same three metrics the rest of the group is using for Ekegusii, so results are comparable across tracks: BLEU, SacreBLEU, and chrF.

In [31]:
bleu_metric = evaluate.load("bleu")
sacrebleu_metric = evaluate.load("sacrebleu")
chrf_metric = evaluate.load("chrf")

def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    if isinstance(predictions, tuple):
        predictions = predictions[0]

    decoded_preds = tokenizer.batch_decode(predictions, skip_special_tokens=True)

    labels = np.where(labels != -100, labels, tokenizer.pad_token_id)
    decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)

    return {
        "bleu": bleu_metric.compute(predictions=decoded_preds, references=[[l] for l in decoded_labels])["bleu"],
        "sacrebleu": sacrebleu_metric.compute(predictions=decoded_preds, references=[[l] for l in decoded_labels])["score"],
        "chrf": chrf_metric.compute(predictions=decoded_preds, references=[[l] for l in decoded_labels])["score"],
    }

print("✅ Metrics loaded: BLEU, SacreBLEU, chrF")

✅ Metrics loaded: BLEU, SacreBLEU, chrF


## Step 14: Training Arguments — 10 Epochs, Batch Training, Epoch Checkpointing to Drive

This is where the three Week 3 requirements are actually enforced:

- `num_train_epochs=10`
- `per_device_train_batch_size` / `per_device_eval_batch_size` — batch training (paired with the collator above)
- `output_dir=CHECKPOINT_DIR` (on Drive) + `save_strategy="epoch"` — a checkpoint is written to Drive automatically after every epoch, no manual saving

Full fine-tuning of a 600M-parameter model needs the memory-saving settings below (`fp16`, gradient checkpointing already enabled on the model, `adafactor` optimizer) to fit on a free-tier T4 GPU. `save_total_limit=3` keeps only the 3 most recent epoch checkpoints on Drive so a full 10-epoch run doesn't blow through free Drive storage — checkpoints are still written every single epoch, older ones are just rotated out automatically. Raise or remove `save_total_limit` if you have the Drive space and want every epoch kept for the report.

**A few settings here specifically for speed** (a full fine-tune of a 600M model on ~25k pairs for 10 epochs is a genuinely multi-hour job on a free T4 — some of that is unavoidable, but the biggest hidden cost is usually the per-epoch *evaluation*, not training itself, because it runs beam-search generation over the whole validation set):

- `generation_num_beams=1` — greedy decoding for the per-epoch progress checks during training. Beam search (beam=4) is 3-4x slower and mostly matters for the final reported quality, not for tracking whether loss/BLEU are moving in the right direction epoch to epoch. Step 18 below explicitly asks for beam=4 on the test set at the end, where quality reporting actually matters.
- `group_by_length=True` — batches similar-length sentences together so batches waste less compute on padding.
- `dataloader_num_workers=2` — loads/tokenizes the next batch on CPU while the GPU is busy, instead of the GPU waiting on the data loader.
- `per_device_train_batch_size=16` with `gradient_accumulation_steps=1` (same effective batch size as before, 16, just achieved as one bigger batch instead of two smaller ones accumulated) — fewer, larger steps use the GPU more efficiently. **If you hit a CUDA out-of-memory error**, drop this back to `per_device_train_batch_size=8, gradient_accumulation_steps=2`.

In [32]:
training_args = Seq2SeqTrainingArguments(
    output_dir=CHECKPOINT_DIR,
    eval_strategy="epoch",
    save_strategy="epoch",              # <- required: auto-checkpoint every epoch
    num_train_epochs=10,                # <- required: 10 epochs
    per_device_train_batch_size=16,     # <- batch training (drop to 8 + accumulation below if you hit OOM)
    per_device_eval_batch_size=16,
    gradient_accumulation_steps=1,      # effective batch size 16; bump to 2 if you drop batch size to 8
    learning_rate=3e-5,                 # full fine-tune -> smaller LR than a LoRA run
    warmup_steps=500,
    weight_decay=0.01,
    predict_with_generate=True,
    generation_max_length=MAX_LEN,
    generation_num_beams=1,             # greedy for fast per-epoch tracking; Step 18 uses beam=4 for the real test-set number
    dataloader_num_workers=2,           # overlap CPU data loading with GPU compute
    fp16=torch.cuda.is_available(),
    optim="adafactor",                  # memory-efficient optimizer, standard for full T5/NLLB fine-tuning
    logging_steps=100,
    save_total_limit=3,                 # keep Drive usage bounded; checkpoints still saved every epoch
    load_best_model_at_end=True,
    metric_for_best_model="sacrebleu",
    greater_is_better=True,
    report_to="none",
    seed=42,
)

print("✅ Training arguments set: 10 epochs, save_strategy='epoch', checkpoints ->", CHECKPOINT_DIR)

✅ Training arguments set: 10 epochs, save_strategy='epoch', checkpoints -> /content/drive/MyDrive/psa-dholuo-mt/nllb-dholuo-checkpoints


## Step 15: Build the Trainer (with Auto-Resume from Drive)

Because checkpoints live on Drive, if the Colab runtime disconnects mid-run you can just re-run the notebook from Step 1 — the cell below detects the most recent checkpoint already on Drive and resumes from there instead of restarting from epoch 0.

In [33]:
# `Seq2SeqTrainer` renamed its `tokenizer=` argument to `processing_class=` in
# newer transformers releases (with `tokenizer=` deprecated but often still
# accepted, and eventually removed). Try the current name first and fall
# back so this cell works across versions.
try:
    trainer = Seq2SeqTrainer(
        model=model,
        args=training_args,
        train_dataset=train_tokenized,
        eval_dataset=val_tokenized,
        data_collator=data_collator,
        processing_class=tokenizer,
        compute_metrics=compute_metrics,
    )
except TypeError:
    trainer = Seq2SeqTrainer(
        model=model,
        args=training_args,
        train_dataset=train_tokenized,
        eval_dataset=val_tokenized,
        data_collator=data_collator,
        tokenizer=tokenizer,
        compute_metrics=compute_metrics,
    )

def find_last_checkpoint(output_dir):
    if not os.path.isdir(output_dir):
        return None
    checkpoints = glob.glob(os.path.join(output_dir, "checkpoint-*"))
    if not checkpoints:
        return None
    checkpoints.sort(key=lambda p: int(p.split("-")[-1]))
    return checkpoints[-1]

last_checkpoint = find_last_checkpoint(CHECKPOINT_DIR)
if last_checkpoint:
    print(f"🔁 Found existing checkpoint on Drive — will resume from: {last_checkpoint}")
else:
    print("🚀 No existing checkpoint found — starting training from epoch 0")

🚀 No existing checkpoint found — starting training from epoch 0


## Step 16: Train

Rough expectations on a free-tier T4: ~25k training pairs, effective batch size 16, 10 epochs — still a multi-hour job, but the greedy per-epoch eval and larger/fewer batches above should meaningfully cut wall-clock time versus beam-search eval with smaller batches. The progress bar shows an `it/s` (iterations per second) figure and a live ETA once it's a few steps in — that's the most reliable way to judge whether a given run is "slow" for your hardware, rather than judging by elapsed minutes alone. Progress (loss + BLEU/SacreBLEU/chrF) prints after every epoch, and a checkpoint lands on Drive after every epoch regardless of how the run ends.

In [34]:
trainer.train(resume_from_checkpoint=last_checkpoint)

Epoch,Training Loss,Validation Loss,Bleu,Sacrebleu,Chrf
1,0.452346,0.504269,0.656364,65.636371,77.777893
2,0.358857,0.448990,0.673168,67.316774,78.990150
3,0.300884,0.430569,0.689133,68.913265,80.018170
4,0.259554,0.420309,0.703842,70.384171,80.908699
5,0.243779,0.417600,0.708188,70.818818,81.494039
6,0.218944,0.422160,0.708352,70.835218,81.498062
7,0.204820,0.423253,0.715749,71.574864,81.919371
8,0.205463,0.424492,0.714699,71.469864,81.959303
9,0.191780,0.427092,0.715864,71.586416,81.998386
10,0.188164,0.427436,0.716872,71.687246,82.114932


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

[transformers] There were missing keys in the checkpoint model loaded: ['model.encoder.embed_tokens.weight', 'model.decoder.embed_tokens.weight', 'lm_head.weight'].


TrainOutput(global_step=16040, training_loss=0.30587215022255954, metrics={'train_runtime': 18846.3547, 'train_samples_per_second': 13.612, 'train_steps_per_second': 0.851, 'total_flos': 4.219383981745766e+16, 'train_loss': 0.30587215022255954, 'epoch': 10.0})

## Step 17: Save the Final Model to Drive

`save_strategy="epoch"` already leaves per-epoch checkpoints on Drive, but it's worth also saving the final trained model to its own clearly-named folder so it's easy to find later (for evaluation, the report, or handing off to a demo).

In [35]:
trainer.save_model(FINAL_MODEL_DIR)
tokenizer.save_pretrained(FINAL_MODEL_DIR)
print(f"✅ Final NLLB-Dholuo model saved to {FINAL_MODEL_DIR}")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

✅ Final NLLB-Dholuo model saved to /content/drive/MyDrive/psa-dholuo-mt/nllb-dholuo-final


## Step 18: Evaluate on the Held-Out Test Set

Per-epoch validation during training used greedy decoding (`num_beams=1`) purely for speed. For the final, reported test-set numbers, this explicitly asks for beam search (`num_beams=4`) — the higher-quality setting the metrics in the Week 3 report should reflect.

In [36]:
test_results = trainer.evaluate(
    eval_dataset=test_tokenized,
    metric_key_prefix="test",
    num_beams=4,
    max_length=MAX_LEN,
)
print("\nTest set results:")
for k, v in test_results.items():
    print(f"  {k}: {v}")

Training Loss,Validation Loss,Epoch,Bleu,Sacrebleu,Chrf
0.188164,0.419267,10,0.717728,71.772807,82.107181



Test set results:
  test_loss: 0.41926702857017517
  test_bleu: 0.7177280656461659
  test_sacrebleu: 71.77280656461659
  test_chrf: 82.10718103194364


## Step 19: Sanity-Check a Few Translations

A quick qualitative look, side-by-side with the reference Dholuo translation, on a handful of test examples.

In [37]:
model.eval()
sample = test_dataset.select(range(min(5, len(test_dataset))))

for ex in sample:
    tokenizer.src_lang = ex["source_lang"]
    inputs = tokenizer(ex["source_text"], return_tensors="pt", truncation=True, max_length=MAX_LEN)
    if torch.cuda.is_available():
        inputs = {k: v.to("cuda") for k, v in inputs.items()}

    with torch.no_grad():
        generated = model.generate(
            **inputs,
            forced_bos_token_id=luo_id,
            max_length=MAX_LEN,
            num_beams=4,
        )
    prediction = tokenizer.batch_decode(generated, skip_special_tokens=True)[0]

    print("Source   :", ex["source_text"])
    print("Reference:", ex["target_text"])
    print("Predicted:", prediction)
    print("-" * 80)

Source   : All schools install CCTV + hire guards by Dec 31 – Ministry directive or face closure!
Reference: Skunde duto onego oket kamera mag CCTV + kendo gikaw jorit e kind Desemba 31 kaluwore gi chik mar Migawo mar somo, ka ok kamano to ibiro loro skundegi!
Predicted: Skunde duto keto CCTV + gitiyo gi jorit kapok Desemba 31 ochopo kaluwore gi chik mar Migawo mar Lendo, ka ok kamano to ibiro loro!
--------------------------------------------------------------------------------
Source   : Shule zote zaweka CCTV + kuajiri walinzi ifikapo Desemba 31 – Maagizo ya Wizara au tutalazimika kufungwa!
Reference: Skunde duto onego oket kamera mag CCTV + kendo gikaw jorit e kind Desemba 31 kaluwore gi chik mar Migawo mar somo, ka ok kamano to ibiro loro skundegi!
Predicted: Skunde duto biro keto CCTV + biro keto jorit e tich kapok Desemba 31 ochopo.
--------------------------------------------------------------------------------
Source   : All TTC students must complete civic education module be